# Celigo hello world

This walkthrough uses the PyLabRobot driver directly; no Celigo vendor application is required. Keep the stage and drawer paths clear whenever motion is enabled.

| Property | Value |
|---|---|
| Device | Nexcelom/Cyntellect Celigo image cytometer |
| Controller transport | FTDI USB, 230400 baud |
| Camera transport | Lumenera SDK on a separate USB device |
| Configuration | Installed Celigo XML/config files plus a standard PyLabRobot plate model |
| Live-tested | Setup, native XYZ/filter homing, drawer, galvos, brightfield, native 2048×2048 ROI and calibrated camera capture |
| Not yet live-tested | Fluorescence, autofocus on cells, triggered acquisition, laser paths |

## Important current gaps

The Lumenera opens at `2464x2056`; `Celigo.setup()` now applies and reads back the centered `2048x2048` ROI required by `CalibrationConfig.xml` (offset `208, 4`). That ROI and a calibrated-size frame have been verified on this hardware. `Celigo.acquire()` still rejects any later geometry mismatch rather than silently producing uncalibrated data.

Image autofocus exists but has not yet been evaluated on a real cell sample. Hardware displacement-sensor autofocus is not implemented. The Corning 3603 has an internal Celigo registration correction; other PyLabRobot plate models use their actual PLR well locations and may need model-specific registration for maximum centering accuracy. Fluorescence imaging, triggered acquisition, and all laser operations remain unverified on hardware. Laser support is disabled by default.

## How it talks

One plain `Celigo` object owns both connections. It talks to the USB-I/O controller through PyLabRobot's FTDI transport, manages the `LumeneraCamera` lifecycle through `liblucamapi`, and loads motor limits, homing profiles, illumination channels, filter mappings, galvo centers, and coordinate transforms from the copied instrument configuration.

## Physical setup

1. Power on the Celigo and clear the stage, objective, and drawer paths.
2. Connect both USB devices: FTDI `0403:6001` for the controller and Lumenera `1724:0645` for the camera.
3. If the devices are attached to another Linux host, export and attach both with USB/IP. For example, on the remote host run `sudo usbip bind -b <busid>`, then locally run `sudo usbip attach -r <host> -b <busid>`.
4. Find the controller's local topology with `lsusb -t`. A USB/IP reattach may change it, so update `controller_usb_address` below.
5. Copy the instrument's `ConfigFiles` directory and identify the matching plate model in `pylabrobot.resources`.

## Imports

The public device API is `Celigo`. It owns the camera internally, and a normal PyLabRobot `Plate` is assigned after construction.

In [ ]:
from pathlib import Path

from pylabrobot.celigo import Celigo
from pylabrobot.resources.corning.plates import Cor_96_wellplate_360ul_Fb

## Configure this instrument

Use the PyLabRobot model matching the physical plate. The driver applies the installed Corning 3603 Celigo registration correction internally; users do not load `.cpr` files. `Celigo.setup()` initializes both the controller and camera and applies the calibrated camera ROI. Setup initializes motors and calibrates/centers the galvos, but deliberately does not home XYZ or the filter wheel.

In [ ]:
config_root = Path("/path/to/Celigo/ConfigFiles")
lucam_sdk = Path("/path/to/liblucamapi.so")
controller_usb_address = "3-2"  # local <bus>-<port>[.<port>...] from lsusb/pyusb

plate = Cor_96_wellplate_360ul_Fb(name="imaging_plate")
celigo = Celigo(
  usb_address=controller_usb_address,
  install_dir=str(config_root),
  lucam_sdk=str(lucam_sdk),
)
celigo.plate = plate

## Connect and initialize

The single setup call opens the controller and camera, performs the binary handshake, reads identity, discovers motors, applies safe outputs, initializes configured motor parameters, and calibrates the galvos.

In [ ]:
await celigo.setup()
celigo.device_info, await celigo.request_motor_map()

## Check camera geometry

Setup should make these equal by applying and verifying the centered native ROI. Calibrated acquisition also checks every returned frame.

In [ ]:
actual_format = (celigo.camera.width, celigo.camera.height)
expected_format = (
  celigo.calibration.image_width_pixels,
  celigo.calibration.image_height_pixels,
)
actual_format, expected_format

## Home the mechanisms

Home Z first for vertical clearance, then X and Y. Each linear routine checks encoder response, negative-limit activation and release, datum establishment, controller-mode restoration, and final encoder arrival. Filter homing uses its encoder index and physical opto tab.

In [ ]:
home_positions = {}
for axis in ("z", "x", "y"):
  home_positions[axis] = await celigo.home(axis)
home_positions["filter"] = await celigo.home_filter_accurate()
home_positions

## Open the drawer

This retracts Z, moves to Y clearance, and drives X/Y to their configured loading limits. Keep hands clear until it finishes. Repeating `open_drawer()` is safe because active destination limits are checked.

In [ ]:
await celigo.open_drawer()

## Load the plate

Place the Corning 3603 plate in the carrier in the instrument's expected orientation. Confirm it is seated flat, then keep clear before running the next cell.

## Close the drawer to A1

The return position is derived from the copied calibration, hardware defaults, standard plate resource, and the driver's internal Celigo registration correction.

In [ ]:
await celigo.close_drawer(plate=plate, well="A1")

## Set a conservative brightfield exposure

The live camera retained settings across reopen. Restarting its stream after changing exposure avoids a stale USB/IP video request. One millisecond at gain 1 was unsaturated during the first hardware check; tune this for the sample.

In [ ]:
await celigo.set_camera_settings(exposure_ms=1.0, gain=1.0, restart=True)

## Acquire one calibrated brightfield image

This single call moves to A1, selects the configured brightfield filter and illumination, centers the calibrated galvos, moves Z to the installed brightfield plane, and captures a geometry-checked frame.

In [ ]:
result = await celigo.acquire(
  "A1",
  "brightfield",
  exposure_ms=1.0,
  gain=1.0,
)
result.z_mm, result.galvo_volts

## Save and inspect the picture

PGM preserves the monochrome pixels without adding an image dependency.

In [ ]:
result.frame.save_pgm("A1-brightfield.pgm")
result.frame.statistics(), result.frame.sharpness(sample_step=8)

## Run image autofocus when the sample has visible structure

Image autofocus scans around the calibrated channel-specific Z plane, scores each frame, and leaves Z at the best plane. It fails closed when there is no measurable contrast or the optimum lies at the scan boundary.

In [ ]:
focused = await celigo.acquire(
  "A1",
  "brightfield",
  exposure_ms=1.0,
  gain=1.0,
  autofocus="image",
)
focused.frame.save_pgm("A1-brightfield-focused.pgm")
focused.focus.z_mm, focused.focus.score

## Change filters and acquire another channel

The channel name selects the installed dichroic position, illumination output and intensity, galvo offsets, and calibrated Z correction. For example:

```python
result = await celigo.acquire(
  "A1",
  "green",
  plate=plate,
  exposure_ms=1.0,
  gain=1.0,
  autofocus="image",
)
result.frame.save_pgm("A1-green.pgm")
```

Available installed names are `brightfield`, `green`, `red`, `blue`, and `far_red`. Fluorescence has not yet been live-validated; begin with conservative exposure and keep `require_lamp_ready=True`.

## Stop safely

Always run this cell, including after an exception. `stop()` aborts controller work, clears analog and digital illumination outputs, closes the camera, and releases FTDI.

In [ ]:
await celigo.stop()